# 01 - Data Exploration

First look at the raw **CAR DETAILS FROM CAR DEKHO** dataset before any cleaning.

Source: CarDekho listings, published on Kaggle by nehalbirla ([kaggle.com/datasets/nehalbirla/car-details-from-cardekho](https://www.kaggle.com/datasets/nehalbirla/car-details-from-cardekho)).

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from data_loader import load_raw_data

pd.set_option('display.max_columns', 20)
df = load_raw_data('../data/raw/CAR DETAILS FROM CAR DEKHO.csv')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

Shape: 4340 rows x 8 columns


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## Structure

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   name           4340 non-null   str  
 1   year           4340 non-null   int64
 2   selling_price  4340 non-null   int64
 3   km_driven      4340 non-null   int64
 4   fuel           4340 non-null   str  
 5   seller_type    4340 non-null   str  
 6   transmission   4340 non-null   str  
 7   owner          4340 non-null   str  
dtypes: int64(3), str(5)
memory usage: 515.2 KB


## Missing values
Zero missing values in every column of the raw file.

In [3]:
df.isnull().sum()

name             0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
dtype: int64

## Exact duplicate rows
The raw file contains a meaningful number of byte-for-byte duplicate listings - handled in `02_data_cleaning.ipynb`.

In [4]:
print('Duplicate rows:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values('name').head(6)

Duplicate rows: 763


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
3912,Ambassador CLASSIC 1500 DSL AC,2005,120000,50000,Diesel,Individual,Manual,Second Owner
4016,Ambassador CLASSIC 1500 DSL AC,2005,120000,50000,Diesel,Individual,Manual,Second Owner
99,Audi A4 2.0 TDI 177 Bhp Premium Plus,2013,1150000,53000,Diesel,Dealer,Automatic,First Owner
2578,Audi A4 2.0 TDI 177 Bhp Premium Plus,2013,1150000,53000,Diesel,Dealer,Automatic,First Owner
554,Audi A4 3.0 TDI Quattro,2013,1580000,86000,Diesel,Dealer,Automatic,First Owner
573,Audi A4 3.0 TDI Quattro,2013,1580000,86000,Diesel,Dealer,Automatic,First Owner


## Numeric summary

In [5]:
df[['year', 'selling_price', 'km_driven']].describe()

,year,selling_price,km_driven
count,4340.000000,4.340000e+03,4340.000000
mean,2013.090783,5.041273e+05,66215.777419
std,4.215344,5.785487e+05,46644.102194
min,1992.000000,2.000000e+04,1.000000
25%,2011.000000,2.087498e+05,35000.000000
50%,2014.000000,3.500000e+05,60000.000000
75%,2016.000000,6.000000e+05,90000.000000
max,2020.000000,8.900000e+06,806599.000000


## Categorical value counts

In [6]:
for col in ['fuel', 'seller_type', 'transmission', 'owner']:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()

--- fuel ---
fuel
Diesel      2153
Petrol      2123
CNG           40
LPG           23
Electric       1
Name: count, dtype: int64

--- seller_type ---
seller_type
Individual          3244
Dealer               994
Trustmark Dealer     102
Name: count, dtype: int64

--- transmission ---
transmission
Manual       3892
Automatic     448
Name: count, dtype: int64

--- owner ---
owner
First Owner             2832
Second Owner            1106
Third Owner              304
Fourth & Above Owner      81
Test Drive Car            17
Name: count, dtype: int64



## Brand (parsed from `name`)
Quick look before this becomes an official engineered feature in `feature_engineering.py`.

In [7]:
df['name'].str.split().str[0].value_counts().head(15)

name
Maruti        1280
Hyundai        821
Mahindra       365
Tata           361
Honda          252
Ford           238
Toyota         206
Chevrolet      188
Renault        146
Volkswagen     107
Skoda           68
Nissan          64
Audi            60
BMW             39
Datsun          37
Name: count, dtype: int64

**Takeaways used later in the pipeline:**
- No missing values, but ~760 exact duplicate rows -> drop in cleaning.
- `name` needs parsing to get a usable `brand` feature; watch for glued tokens like `OpelCorsa` and two-word manufacturers like `Land Rover`.
- `selling_price` is strongly right-skewed -> log-transform the target for modelling.